# lineup — model stages on a free cloud GPU

This notebook runs the GPU stages of the benchmark on a free Colab or Kaggle T4 (16 GB). It clones the repository, builds test cases, loads Qwen2.5-7B-Instruct in 4-bit, and runs generation, the leave-one-out oracle, the attribution methods, the scorer, and the abstention experiment end to end — then exports the results for the explorer app.

This is a small, fast demo (a couple dozen cases). The leave-one-out oracle re-runs the model once per chunk, so it is the slowest stage — every model loop shows a progress bar. For a full run, raise `limit` and use the command-line scripts.

Before running, set the runtime to a GPU: **Runtime → Change runtime type → T4 GPU** (on Kaggle, enable the GPU accelerator and Internet in the sidebar).

In [ ]:
import sys, os
if os.path.isdir('/content/LINEUP'):
    !cd /content/LINEUP && git pull -q
else:
    !git clone -q https://github.com/santoshcheethiralame-dot/LINEUP
%cd /content/LINEUP
!pip install -q -e . bitsandbytes
if '/content/LINEUP/src' not in sys.path:
    sys.path.insert(0, '/content/LINEUP/src')
import lineup
print('lineup', lineup.__version__, 'ready')

## Load Qwen2.5-7B in 4-bit

The first load downloads about 5 GB of 4-bit weights. Answers are short, so 24 new tokens is plenty and keeps generation quick.

In [ ]:
from lineup.backends import TransformersModel
from lineup.config import DEFAULT_MODEL, set_seed

set_seed()
model = TransformersModel(DEFAULT_MODEL, load_in_4bit=True, max_new_tokens=24)
print("loaded", DEFAULT_MODEL)

## Build the cases and run the model

Small context (`k=6`) keeps the per-case leave-one-out cost down. Scenario construction is model-free and deterministic; the same seed reproduces the same benchmark anywhere.

In [ ]:
from tqdm.auto import tqdm
from lineup.data.hotpotqa import load_examples
from lineup.data.substitution import build_answer_pool
from lineup.data.scenario import ScenarioBuilder
from lineup.data.misleading import substitution_check
from lineup.correctness import LLMJudge
from lineup.generation import generate_and_judge

examples = list(load_examples("validation", limit=24))
pool = build_answer_pool(examples)
builder = ScenarioBuilder(answer_pool=pool, k=6, seed=0)
judge = LLMJudge(model)

scenarios, originals = [], []
for example in tqdm(examples, desc="generate"):
    if substitution_check(example):
        continue
    scenario = builder.build(example)
    if scenario is None:
        continue
    scenarios.append(scenario)
    originals.append(generate_and_judge(model, scenario, llm_judge=judge))

wrong = [r for r in originals if not r.is_correct]
print(f"built {len(scenarios)} cases, {len(wrong)} answered wrongly")

## Stage 4 — counterfactual oracle

Leave-one-out labels the wrong cases (a correct answer needs no culprit hunt, so it gets a trivial record). This is the slow stage; the bar shows progress.

In [ ]:
from lineup.data.schema import CaseRoles
from lineup.oracle import leave_one_out

role_cases = []
for scenario, original in tqdm(list(zip(scenarios, originals)), desc="oracle"):
    if original.is_correct:
        role_cases.append(CaseRoles(scenario.qid, scenario.question, scenario.gold_answer, original.model_answer, True, []))
    else:
        role_cases.append(leave_one_out(model, scenario, original))
print(f"labeled {len(role_cases)} cases")

## Stage 5 — methods under test

Run each attribution method on every case.

In [ ]:
from lineup.methods import ContextCite, LexicalSimilarity, LLMJudgeCulprit, SingleChunkSupport, run_method

methods = [ContextCite(n_ablations=8, seed=0), LexicalSimilarity(), LLMJudgeCulprit(), SingleChunkSupport()]
predictions = []
for scenario, original in tqdm(list(zip(scenarios, originals)), desc="methods"):
    for method in methods:
        predictions.append(run_method(method, model, scenario, original.model_answer))
print(f"{len(predictions)} predictions")

## Stage 6 — scorer

Grade the methods on the wrong cases. A salience-based method should show a high `misleading-as-culprit` rate and a low `culprit>misleading` win-rate.

In [ ]:
from lineup.scoring import score_predictions

wrong_cases = [case for case in role_cases if not case.original_correct]
for report in score_predictions(wrong_cases, predictions):
    top1 = f"{report.top1_culprit_accuracy:.2f}" if report.top1_culprit_accuracy is not None else "n/a"
    win = f"{report.culprit_over_misleading_winrate:.2f}" if report.culprit_over_misleading_winrate is not None else "n/a"
    print(f"{report.method:18s} top1={top1} misleading-as-culprit={report.misleading_as_culprit_rate:.2f} culprit>misleading={win}")

## Stage 7 — does it matter?

Selective QA over all cases: can a confidence signal tell correct answers from wrong ones? Compare the model's own confidence, each method's attribution decisiveness, and the oracle upper bound by AUROC.

In [ ]:
from lineup.downstream import evaluate_abstention

for report in evaluate_abstention(originals, predictions, role_cases):
    auroc = f"{report.auroc:.2f}" if report.auroc is not None else "n/a"
    print(f"{report.signal:22s} n={report.n} correct={report.n_correct} AUROC={auroc}")

## Export for the explorer app

Write the four JSONL files this run produced and download them as `lineup_outputs.zip`. Unzip it, then run the app locally (`streamlit run app/app.py`) and point the sidebar **data directory** at the unzipped folder to drive a demo on these real results.

In [ ]:
import shutil
from pathlib import Path
from lineup.data.serialization import write_generations, write_predictions, write_roles, write_scenarios

out = Path("outputs")
out.mkdir(exist_ok=True)
write_scenarios(out / "scenarios.jsonl", scenarios)
write_generations(out / "generations.jsonl", originals)
write_roles(out / "roles.jsonl", role_cases)
write_predictions(out / "predictions.jsonl", predictions)
shutil.make_archive("lineup_outputs", "zip", out)
print(f"wrote {len(scenarios)} cases to outputs/ and lineup_outputs.zip")

try:
    from google.colab import files
    files.download("lineup_outputs.zip")
except Exception:
    print("download lineup_outputs.zip from the file browser on the left")

## Notes

- This demo runs a couple dozen cases with a small context so it finishes in minutes; the numbers stabilise with a larger `limit`.
- 4-bit (nf4) keeps the 7B model within a 16 GB T4; fp16 compute is used because the T4 has no native bfloat16.
- Run **all** model stages on one machine and one pinned model revision — greedy decoding is deterministic per machine, but log-probabilities drift across GPUs and precisions, so the labels must come from a single box.